In [1]:
from groq import Groq
import base64
import os
from pathlib import Path
import fitz  # PyMuPDF

# Make sure API key is set
# os.environ["GROQ_API_KEY"] = "YOUR_API_KEY"

os.environ["GROQ_API_KEY"] = "YOUR API KEY"
client = Groq(api_key=os.environ.get("GROQ_API_KEY"))


In [2]:
def ocr_image_with_groq(image_path):
    with open(image_path, "rb") as image_file:
        base64_image = base64.b64encode(image_file.read()).decode("utf-8")

    response = client.chat.completions.create(
        model="meta-llama/llama-4-scout-17b-16e-instruct",
        messages=[
            {
                "role": "user",
                "content": [
                    {"type": "text", "text": "Extract all readable text from this medical report image."},
                    {
                        "type": "image_url",
                        "image_url": {
                            "url": f"data:image/jpeg;base64,{base64_image}",
                        },
                    },
                ],
            }
        ],
    )

    text = response.choices[0].message.content

    if not text or text.strip() == "":
        raise ValueError("OCR failed")

    return text

In [3]:
def ocr_pdf(pdf_path):
    doc = fitz.open(pdf_path)
    full_text = ""

    for page_no in range(len(doc)):
        page = doc.load_page(page_no)
        pix = page.get_pixmap(dpi=300)

        temp_img = f"temp_{page_no}.png"
        pix.save(temp_img)

        full_text += "\n" + ocr_image_with_groq(temp_img)

        os.remove(temp_img)

    return full_text

In [6]:
RAW_DIR = "data/raw_reports"
OUT_DIR = "data/extracted_text"

os.makedirs(OUT_DIR, exist_ok=True)

for name in os.listdir(RAW_DIR):
    in_path = os.path.join(RAW_DIR, name)

    # 🔴 SKIP folders like .ipynb_checkpoints
    if not os.path.isfile(in_path):
        continue

    out_path = os.path.join(OUT_DIR, Path(name).stem + ".txt")

    print("Processing:", name)

    if name.lower().endswith(".pdf"):
        text = ocr_pdf(in_path)
    elif name.lower().endswith((".png", ".jpg", ".jpeg")):
        text = ocr_image_with_groq(in_path)
    else:
        print("Skipping unsupported file:", name)
        continue

    with open(out_path, "w", encoding="utf-8") as f:
        f.write(text)

print("✅ Batch OCR completed successfully")


Processing: report1.pdf
Processing: report10.png
Processing: report2.png
Processing: report3.png
Processing: report4.png
Processing: report5.pdf
Processing: report6.pdf
Processing: report7.png
Processing: report8.png
Processing: report9.jpg
✅ Batch OCR completed successfully


In [7]:
for f in os.listdir("data/extracted_text"):
    print(f, "→ size:", os.path.getsize("data/extracted_text/" + f))

report1.txt → size: 36818
report10.txt → size: 2965
report2.txt → size: 2992
report3.txt → size: 1684
report4.txt → size: 2144
report5.txt → size: 43240
report6.txt → size: 2803
report7.txt → size: 2435
report8.txt → size: 1163
report9.txt → size: 2528
